In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import os


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  
])


In [ ]:
dataset = datasets.ImageFolder("../../data/specific_cancer_dataset/train", transform=transform)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])


In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


In [ ]:
class RouterModel(nn.Module):
    def __init__(self, num_classes:int):
        super(RouterModel, self).__init__()
        self.base_model = models.resnet18(pretrained=True)

        in_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.base_model(x)

In [ ]:
num_classes = len(dataset.classes)
print("Detected classes:", dataset.classes)
print("Number of classes:", num_classes)

In [ ]:
router_model = RouterModel(num_classes=num_classes)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
router = router_model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(router.parameters(), lr=1e-4)

In [ ]:
for epoch in range(5):
    router.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = router(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
os.makedirs("saved_models", exist_ok=True)  
torch.save(router.state_dict(), "saved_models/router_model.pth")
print("Model saved to saved_models/router_model.pth")